# Spark MLlib – Preprocesamiento de Datos

**Curso:** Computación de Alto Desempeño – MLlib Spark  
**Autor:** Santiago Gil Gallego (Sgg)  
**Fecha:** 27 de noviembre de 2025  

**Objetivo del cuaderno:**  
Este cuaderno realiza Clustering (K-means) sobre el dataset Iris preprocesado, se cálcula el valor óptimo de k y se usa el método de silueta para verificar su precisión.


In [1]:
import findspark
findspark.init()

from pyspark import SparkConf
from pyspark.sql import SparkSession, SQLContext

configuraSgg = (
    SparkConf()
    .set("spark.scheduler.mode", "FAIR")
    .set("spark.executor.cores", "1")
    .set("spark.executor.memory", "4G")
    .set("spark.cores.max", "2")
    .setMaster("spark://10.43.100.121:7077")
)
configuraSgg.setAppName("hpcsparkSgg_nosupervisado_cluster")

sparkSgg = SparkSession.builder.config(conf=configuraSgg).getOrCreate()
sqlContext = SQLContext(sparkContext=sparkSgg.sparkContext,
                        sparkSession=sparkSgg)

print("MASTER ACTUAL:", sparkSgg.sparkContext.master)

df_unsup = sparkSgg.read.parquet("data/unsupervised/preprocessed_sgg")
df_unsup.printSchema()
df_unsup.show(5)



Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/11/26 16:44:22 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
25/11/26 16:44:22 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.


MASTER ACTUAL: spark://10.43.100.121:7077


root
 |-- features: vector (nullable = true)



+--------------------+
|            features|
+--------------------+
|[-1.0153732795881...|
|[-0.8950147921906...|
|[0.66964554397658...|
|[-0.5339393299982...|
|[-0.2932223552032...|
+--------------------+
only showing top 5 rows



In [2]:
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator

ks = [2, 3, 4, 5]
results = []

for k in ks:
    kmeans = KMeans(k=k, seed=42, featuresCol="features")
    model = kmeans.fit(df_unsup)
    preds = model.transform(df_unsup)
    
    evaluator = ClusteringEvaluator(
        featuresCol="features",
        predictionCol="prediction",
        metricName="silhouette",
        distanceMeasure="squaredEuclidean"
    )
    silhouette = evaluator.evaluate(preds)
    
    results.append((k, silhouette))
    print(f"k={k}, silhouette={silhouette:.4f}")


25/11/26 16:44:47 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS


k=2, silhouette=0.7721


k=3, silhouette=0.6465
k=4, silhouette=0.5884
k=5, silhouette=0.5163


In [3]:
best_k, best_sil = max(results, key=lambda x: x[1])
print("Mejor k:", best_k, "con silhouette:", best_sil)

kmeans_best = KMeans(k=best_k, seed=42, featuresCol="features")
model_best = kmeans_best.fit(df_unsup)
predictions_best = model_best.transform(df_unsup)

predictions_best.groupBy("prediction").count().show()

centers = model_best.clusterCenters()
for i, center in enumerate(centers):
    print(f"Centroide cluster {i}: {center}")


Mejor k: 2 con silhouette: 0.772108980146272
+----------+-----+
|prediction|count|
+----------+-----+
|         1|   50|
|         0|   99|
+----------+-----+

Centroide cluster 0: [ 0.50916756 -0.42625757  0.6533846   0.62823572]
Centroide cluster 1: [-1.00815177  0.84398999 -1.2937015  -1.24390673]


## Conclusiones

- Se entrenó un modelo de k medianas para el dataset iris. Este reveló que el procedimiento típico no es suficiente y se deberían explorar otras opciones de ingeniería de características, se usaron valores de k entre 2 y 5, y se calculó el valor de silhouette para cada uno, resultando así:
- El modelo obtuvo aproximadamente:
  - **k=2:** `silhouette=0.7721`
  - **k=3:** `silhouette=0.6465`
  - **k=4:** `silhouette=0.5884`
  - **k=5:** `silhouette0.5163`
- Estos resultados indican que el proceso de extracción de caracterísiticas no tiene un buen desempeño para este dataset clásico.
- Como trabajo futuro se podrían:
  - Juntar valores para crear mejores características.
  - Analizar otros modelos de clasificación.